In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
#         print(os.path.join(dirname, filename))
        break
        

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.2 MB/s eta 0:00:00a 0:00:01


In [4]:
import shutil
import os
import torch

from base64 import b64encode
from ultralytics import YOLO
from IPython.display import Image, HTML
from tqdm import tqdm

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
print("Is CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
print("CUDA Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Is CUDA available: True
Number of GPUs: 2
CUDA Device Name: Tesla T4


In [6]:
!nvidia-smi

Sat Feb  7 04:49:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
os.listdir("/kaggle/input")


['pothole-dataset-1']

In [7]:
data_yml_path = '/kaggle/input/pothole-dataset-1/data.yaml'
results_path = '/kaggle/working/runs/detect/'
weights = 'yolov8n.pt' # https://docs.ultralytics.com/models/
batch_size = 32

#### Training YOLO with Distributed Data Parallel (DDP)

This script provides a Python-based approach to train the YOLO model using the Ultralytics library with multi-GPU support through PyTorch's Distributed Data Parallel (DDP).

#### Parameters Explained

- **batch_size**: Defines the batch size for each training iteration. Typical values are 32, 64, or 128, but can be adjusted based on GPU memory capacity.
- **data_path**: Specifies the path to the data configuration file, which includes dataset paths and training parameters.
- **weights**: The model's starting weights, usually from a pre-trained YOLO model.
- **devices**: Comma-separated GPU IDs for distributed training; adjust based on available GPUs.

#### Additional Resources

- **[Ultralytics Documentation](https://docs.ultralytics.com/)**: Provides detailed information on YOLO model usage, parameters, and customization options.
- **[PyTorch DDP Documentation](https://pytorch.org/docs/stable/distributed.html)**: Official documentation for using DistributedDataParallel in multi-GPU training.

In [8]:
def train_yolo_with_multi_gpu(
    data_path: str = data_yml_path,
    epochs: int = 150,
    batch_size: int = batch_size,
    weights: str = weights ,
    imgsz: int = 640,
    devices: str = '0,1', 
    **kwargs
) -> None:
    
   
    model = YOLO(weights)
    
    
    model.train(
        data=data_path,
        epochs=epochs,
        batch=batch_size,
        imgsz=imgsz,
        device=devices,  
        **kwargs
    )

# Jalankan pelatihan
train_yolo_with_multi_gpu()

Ultralytics 8.4.12 🚀 Python-3.10.14 torch-2.4.0 CUDA:0 (Tesla T4, 14913MiB)
                                                CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/pothole-dataset-1/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=N

/opt/conda/lib/python3.10/site-packages/ultralytics/utils/metrics.py:75: UserWarning: Specified kernel cache directory could not be created! This disables kernel caching. Specified directory is /root/.cache/torch/kernels. This warning will appear only once per process. (Triggered internally at /usr/local/src/pytorch/aten/src/ATen/native/cuda/jit_utils.cpp:1442.)
  inter = (torch.min(a2, b2) - torch.max(a1, b1)).clamp_(0).prod(2)


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 2.7it/s 7.5s0.3s
                   all       1219       3104      0.483      0.394      0.371      0.156

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/150      2.52G      1.669      1.846      1.499         53        640: 100% ━━━━━━━━━━━━ 234/234 2.2it/s 1:45<0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 3.5it/s 5.8s0.3s
                   all       1219       3104      0.449      0.331      0.317      0.137

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/150      2.52G      1.734      1.741      1.547         44        640: 100% ━━━━━━━━━━━━ 234/234 2.4it/s 1:37<0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 3.4it/s 5.9s0.3s
                   all 

In [ ]:
# def train_yolo_with_single_gpu(     
#     data_path: str=data_yml_path,
#     epochs: int=100,
#     batch_size: int=batch_size,
#     weights: str=weights,
#     imgsz: int=640,
#     device: str='0', 
#     **kwargs
    
#     ) -> None:
    
#     model = YOLO(weights)
    
#     model.train(
#         data=data_path,
#         epochs=epochs,
#         batch=batch_size,
#         imgsz=imgsz,
#         device=device
#     )

# train_yolo_with_single_gpu()

In [ ]:
# def train_yolo_with_cpu(     
#     data_path: str=data_yml_path,
#     epochs: int=100,
#     batch_size: int=batch_size,
#     weights: str=weights,
#     imgsz: int=640,
#     device: str='cpu', 
#     **kwargs
    
#     ) -> None:
    
#     model = YOLO(weights)
    
#     model.train(
#         data=data_path,
#         epochs=epochs,
#         batch=batch_size,
#         imgsz=imgsz,
#         device=device
#     )

# train_yolo_with_cpu()

In [ ]:
Image(results_path + "train/confusion_matrix.png", width=600)

In [ ]:
Image(results_path + "train/results.png", width=600)

In [ ]:
Image(results_path + "train/val_batch2_labels.jpg", width=800)

In [ ]:
Image(results_path + "train/val_batch2_pred.jpg", width=800)

In [9]:
def validation_yolo_single_gpu(   
    data_path: str=data_yml_path,
    weights: str='/kaggle/working/runs/detect/train/weights/best.pt',
    imgsz: int=640,
    batch_size: int=batch_size,
    conf: float=0.25,
    iou: float=0.6,
    device: str='0', 
    **kwargs
    
    ):
    
    model = YOLO(weights)
    
    validation_results = model.val(
        data=data_path, 
        imgsz=imgsz, 
        batch=batch_size, 
        conf=conf, 
        iou=iou, 
        device=device,
        **kwargs
    )
    
    return validation_results

validation_results = validation_yolo_single_gpu()

Ultralytics 8.4.12 🚀 Python-3.10.14 torch-2.4.0 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 92.0±41.9 MB/s, size: 46.2 KB)
val: Scanning /kaggle/input/pothole-dataset-1/valid/labels... 1219 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1219/1219 527.3it/s 2.3s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/pothole-dataset-1/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 39/39 3.3it/s 11.7s0.3s
                   all       1219       3104      0.886      0.751      0.855      0.643
Speed: 0.9ms preprocess, 4.1ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/val


In [10]:
print(validation_results.box.map)  # mAP50-95
print(validation_results.box.map50)  # mAP50
print(validation_results.box.map75)  # mAP75
print(validation_results.box.maps)  # list of mAP50-95 for each category

0.6434513840057965
0.8553158245878386
0.7172994626260101
[    0.64345]


In [11]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")

model.export(format="onnx", imgsz=640, simplify=True)


Ultralytics 8.4.12 🚀 Python-3.10.14 torch-2.4.0 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/234.2 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.2/234.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/300.5 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/300.5 MB 81.6 MB/s eta 0:00:04
   ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/300.5 MB 281.1 MB/s eta 0:00:02
   ━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/300.5 MB 316

'runs/detect/train/weights/best.onnx'

Fine Tuning the model (optional) 

In [ ]:
def train_yolo_with_multi_gpu(
    data_path: str = data_yml_path,
    epochs: int = 60,
    batch_size: int = 8,
    weights: str = weights ,
    imgsz: int = 640,
    devices: str = '0,1', 
    **kwargs
) -> None:
    
   
    model = YOLO("runs/detect/train/weights/best.pt")
    
    
    model.train(
        data=data_path,
        epochs=epochs,
        batch=batch_size,
        imgsz=imgsz,
        device=devices,  
        **kwargs
    )

# Jalankan pelatihan
train_yolo_with_multi_gpu()

In [ ]:
def validation_yolo_single_gpu(   
    data_path: str=data_yml_path,
    weights: str='/kaggle/working/runs/detect/train2/weights/best.pt',
    imgsz: int=640,
    batch_size: int=batch_size,
    conf: float=0.25,
    iou: float=0.6,
    device: str='0', 
    **kwargs
    
    ):
    
    model = YOLO(weights)
    
    validation_results = model.val(
        data=data_path, 
        imgsz=imgsz, 
        batch=batch_size, 
        conf=conf, 
        iou=iou, 
        device=device,
        **kwargs
    )
    
    return validation_results

validation_results = validation_yolo_single_gpu()

In [ ]:
print(validation_results.box.map)  # mAP50-95
print(validation_results.box.map50)  # mAP50
print(validation_results.box.map75)  # mAP75
print(validation_results.box.maps)  # list of mAP50-95 for each category

In [1]:
def prediction_yolo_single_gpu(   
    source: str='/kaggle/input/potholes-detection-yolov8/sample_video.mp4',
    weights: str='/kaggle/working/runs/detect/train/weights/best.pt',
    conf: float=0.25,
    save: bool=True,
    **kwargs
    ):
    
    model = YOLO(weights)

    prediction_results = model.predict(
        source=source,  # path ke data
        conf=conf,  # confidence threshold
        save=save,  # simpan hasil prediksi
        **kwargs
    )
    
    return prediction_results

prediction_results = prediction_yolo_single_gpu()

NameError: name 'YOLO' is not defined